# CLP-SNN Hyperparameter Sweep — Threshold × g_inc (Supplemental)

Sweeps `threshold` ∈ {0.70, 0.75, 0.80} and `g_inc` ∈ {0.1, 0.2, 0.5, 1.0} for
the CLP-SNN float and int variants. Visualises the stability boundary as a heatmap
of mean final accuracy (mean over 3 seeds).

Run the sweep first (or set `RUN_EXPERIMENT = True`):
```bash
python experiments/clp_snn_threshold_g_inc_sweep.py
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

RESULTS_DIR = REPO / "experiments" / "results" / "clp_snn_threshold_g_inc_sweep"
IMAGES_DIR  = REPO / "images"
IMAGES_DIR.mkdir(exist_ok=True)

RUN_EXPERIMENT = False  # set True to re-run

In [ ]:
if RUN_EXPERIMENT:
    import subprocess
    subprocess.run([sys.executable,
                    str(REPO / "experiments" / "clp_snn_threshold_g_inc_sweep.py")],
                   check=True)

## Load results

In [ ]:
df = pd.read_csv(RESULTS_DIR / "results_flat.csv")
print(df.columns.tolist())
df.head()

## Heatmaps: mean final accuracy for each variant

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "savefig.dpi": 600,
})

variants = df["variant"].unique()
fig, axes = plt.subplots(1, len(variants), figsize=(3.5 * len(variants), 2.5))
if len(variants) == 1:
    axes = [axes]

for ax, var in zip(axes, variants):
    sub = df[df["variant"] == var]
    pivot = sub.groupby(["threshold", "g_inc"])["final_acc"].mean().unstack("g_inc") * 100
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".1f", cmap="viridis",
                vmin=0, vmax=100, linewidths=0.3, cbar_kws={"label": "Accuracy (%)"})
    ax.set_title(var)
    ax.set_xlabel("g_inc")
    ax.set_ylabel("threshold")

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(IMAGES_DIR / f"heatmap_threshold_ginc.{ext}", format=ext, bbox_inches="tight")
plt.show()
print("Saved to", IMAGES_DIR)

## Summary: best (threshold, g_inc) per variant

In [ ]:
summary = (
    df.groupby(["variant", "threshold", "g_inc"])["final_acc"]
    .mean()
    .reset_index()
    .sort_values("final_acc", ascending=False)
    .groupby("variant")
    .first()
    .reset_index()
)
summary["final_acc"] = (summary["final_acc"] * 100).round(2)
summary.columns = ["Variant", "Best threshold", "Best g_inc", "Mean acc (%)"]
summary